# DME Express — SQL Explorer & Table Structure Probe

**Purpose.** A scratch bench for looking at the DMEEXPRESS warehouse: list tables, read
column definitions, profile a column's shape, and preview rows — without touching the
production dashboard notebook. Nothing here produces a published metric. Use it to answer
"what *is* this table?" before you write an extract in `ops_dashboard` or `tech_workload`.

**Read-only, by construction.** `run_query()` refuses anything that is not a single
SELECT/WITH statement (CLAUDE.md rule 4). Structure is read from `INFORMATION_SCHEMA`
views rather than `sp_help`/`sp_columns`, because those are stored-procedure calls and
`EXEC` is refused — the views return the same facts through a plain SELECT.

**Outputs are never committed** (rule 3). Row-level previews stay on your machine.
`PREVIEW_ROWS` is small on purpose, and §7 is the only place raw rows are shown.

---

### Changelog

- **v1.0.0 (2026-08-15)** — initial. Catalog helpers (§4), column profiler (§5),
  `PLC_EMPLOYEE_HOURS` structure section (§6), and the payroll **week-boundary probe**
  (§8) that determines empirically which weekday the Paylocity week starts on, rather
  than assuming it.

### How to use

Run §0–§5 once (setup + helpers). After that every section is independent — jump to the
one you need. §9 is a blank scratch cell; put throwaway queries there, not above.

## Cell 1 — INSTALL DEPENDENCIES

In [ ]:
%pip install pandas python-dotenv pypyodbc azure-identity azure-keyvault-secrets

## Cell 2 — IMPORTS

In [ ]:
# Deliberately lighter than the dashboard: no matplotlib, no rapidfuzz. This notebook
# reads structure, it does not render anything.
import os, re, time, warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import pypyodbc as odbc
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 44)
print(f'pandas {pd.__version__} | numpy {np.__version__}')

## Cell 3 — CONFIGURATION *(every adjustable parameter lives here)*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
# Connection — identical to ops_dashboard v1.3.0 Cell 4.1. The Key Vault holds a BARE
# hostname; the 'tcp:...,1433' wrapper is required (audit B4 — a bare host times out
# with [08001] TCP Provider: Timeout error [258] on a cold connect).
KEY_VAULT_URI   = 'https://dmee-keyvault.vault.azure.net/'
HOSTNAME_SECRET = 'DataWarehouseHostname'
SQL_USERNAME    = 'anewton-ro'          # secret NAME is the login; its VALUE is the password
DATABASE_NAME   = 'DMEEXPRESS'
DRIVER_NAME     = 'ODBC Driver 18 for SQL Server'

# What we are looking at by default.
TARGET_TABLE  = 'PLC_EMPLOYEE_HOURS'    # §6-§8 profile this table
TARGET_SCHEMA = 'dbo'

# Window used by the profiling and week-boundary sections. Keep it aligned with the
# dashboard so anything you learn here transfers without a rebasing argument.
FILTER_START = '2025-01-01'
AS_OF_DATE   = datetime.now().date() - timedelta(days=1)
FILTER_END   = AS_OF_DATE.strftime('%Y-%m-%d')

# Profiler knobs.
PREVIEW_ROWS      = 5     # rows shown by preview(); keep small — outputs are not committed
TOP_VALUES_N      = 12    # distinct values listed per categorical column
PROFILE_MAX_COLS  = 40    # guard against profiling a 200-column table by accident
SAMPLE_ROWS_LIMIT = None  # e.g. 200000 to profile a TOP-N sample of a very large table

print(f'Target: {TARGET_SCHEMA}.{TARGET_TABLE} | window {FILTER_START} -> {FILTER_END}')

## Cell 4.1 — SECURE SQL CONNECTION (Azure Key Vault)

In [ ]:
# TEACHING NOTE — why Key Vault and not a .env file: the credential never lands on disk
# and never enters notebook globals as plaintext beyond this cell. CLAUDE.md rule 5 puts
# ~/.dme-secrets and .env files off-limits to the agent entirely; Key Vault sidesteps the
# question. DefaultAzureCredential picks up your existing Azure login (az login / VS Code
# / managed identity) — if it prompts, that is expected on a cold machine.
kv = SecretClient(vault_url=KEY_VAULT_URI, credential=DefaultAzureCredential())
SERVER_NAME  = f'tcp:{kv.get_secret(HOSTNAME_SECRET).value},1433'
_SQL_PASSWORD = kv.get_secret(SQL_USERNAME).value

sql_conn = odbc.connect(
    f'DRIVER={{{DRIVER_NAME}}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};'
    f'UID={SQL_USERNAME};PWD={_SQL_PASSWORD};'
    f'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
    f'ApplicationIntent=ReadOnly;'   # routing hint only — NOT a security boundary
)
del _SQL_PASSWORD   # don't leave the password in notebook globals
print('DB connection established (read-only intent).')

## Cell 4.2 — READ-ONLY QUERY RUNNER *(lifted from ops_dashboard v1.3.0 Cell 4.2)*

In [ ]:
# LIFTED VERBATIM from ops_dashboard v1.3.0 Cell 4.2 — re-sync if that changes.
# This is the in-process enforcement CLAUDE.md rule 4 refers to. It is a guard against an
# accident in a notebook holding a live production connection, not a security boundary;
# the real boundary is the server-side grant on 'anewton-ro' (db_datareader only).
_READ_ONLY_STARTS = ('select', 'with')

def run_query(query, label='', verbose=False):
    """Run a READ-ONLY query and return a DataFrame. Raises before touching the
    connection if the statement is anything other than a single SELECT/WITH."""
    _q = re.sub(r'--[^\n]*', ' ', query)             # strip line comments
    _q = re.sub(r'/\*.*?\*/', ' ', _q, flags=re.S)   # strip block comments
    _q_clean = _q.strip().lstrip('(').lstrip()
    if not _q_clean.lower().startswith(_READ_ONLY_STARTS):
        raise ValueError(f'run_query is read-only: a query must begin with SELECT or WITH '
                         f'(label={label!r}, got {_q_clean[:60]!r}).')
    _forbidden = re.findall(r'(?<![\w.])(insert|update|delete|merge|drop|truncate|alter|create|'
                            r'grant|revoke|exec|execute|sp_\w+|xp_\w+)(?![\w.])', _q, flags=re.I)
    if _forbidden:
        raise ValueError(f'run_query is read-only: refusing statement containing '
                         f'{sorted(set(w.lower() for w in _forbidden))} (label={label!r}).')
    if ';' in _q_clean.rstrip().rstrip(';'):
        raise ValueError(f'run_query is read-only: one statement per call (label={label!r}).')
    if verbose:
        print(f'Query: {label}\n{query}')
    t0 = time.time()
    cur = sql_conn.cursor()
    cur.execute(query)
    rows = cur.fetchall()
    cols = [c[0].lower() for c in cur.description]
    df = pd.DataFrame(rows, columns=cols)
    if label:
        print(f'  {label}: {len(df):,} rows  ({time.time() - t0:.1f}s)')
    return df

def qi(name):
    """Quote an identifier for interpolation. Rejects anything that is not a plain
    identifier, so a table name can never smuggle SQL into an f-string."""
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_ ]*', str(name) or ''):
        raise ValueError(f'unsafe identifier: {name!r}')
    return f'[{name}]'

## Cell 5 — CATALOG HELPERS

*Teaching note.* `INFORMATION_SCHEMA` is the ANSI-standard, SELECT-able view over the
catalog. `sp_help` would be friendlier to type but it is a stored procedure, and
`run_query` refuses `EXEC` — so the views are both the compliant route and the portable
one. `sys.partitions` gives an approximate row count for free (no table scan), which is
what you want when you are deciding whether a table is worth opening at all.

In [ ]:
def list_tables(like=None, schema=None):
    """Tables/views whose name matches `like` (SQL LIKE syntax, % wildcards)."""
    w = ["TABLE_CATALOG IS NOT NULL"]
    if like:   w.append(f"TABLE_NAME LIKE '{like}'")
    if schema: w.append(f"TABLE_SCHEMA = '{schema}'")
    return run_query(f"""
SELECT TABLE_SCHEMA AS [schema], TABLE_NAME AS [table], TABLE_TYPE AS [type]
FROM INFORMATION_SCHEMA.TABLES
WHERE {' AND '.join(w)}
ORDER BY TABLE_SCHEMA, TABLE_NAME
""", f'Tables LIKE {like!r}')

def table_sizes(like=None):
    """Approximate row counts from catalog metadata — no scan, instant on huge tables.
    'Approximate' matters: it is maintained by the engine and can lag a bulk load."""
    w = f"AND t.name LIKE '{like}'" if like else ''
    return run_query(f"""
SELECT s.name AS [schema], t.name AS [table], SUM(p.rows) AS approx_rows
FROM sys.tables AS t
JOIN sys.schemas AS s ON s.schema_id = t.schema_id
JOIN sys.partitions AS p ON p.object_id = t.object_id AND p.index_id IN (0, 1)
WHERE 1 = 1 {w}
GROUP BY s.name, t.name
ORDER BY SUM(p.rows) DESC
""", 'Approx row counts')

def describe_table(table=None, schema=None):
    """Column definitions: type, nullability, width, precision. The 'what is this table'
    answer."""
    table  = table or TARGET_TABLE
    schema = schema or TARGET_SCHEMA
    return run_query(f"""
SELECT ORDINAL_POSITION AS pos, COLUMN_NAME AS column_name, DATA_TYPE AS data_type,
       CHARACTER_MAXIMUM_LENGTH AS max_len, NUMERIC_PRECISION AS num_precision,
       NUMERIC_SCALE AS num_scale, IS_NULLABLE AS is_nullable,
       COLUMN_DEFAULT AS column_default
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = '{table}' AND TABLE_SCHEMA = '{schema}'
ORDER BY ORDINAL_POSITION
""", f'Columns of {schema}.{table}')

def preview(table=None, schema=None, n=None, where=None):
    """First n rows. Keep n small — notebook outputs are never committed (rule 3)."""
    table  = table or TARGET_TABLE
    schema = schema or TARGET_SCHEMA
    n = n or PREVIEW_ROWS
    w = f'WHERE {where}' if where else ''
    return run_query(f'SELECT TOP {int(n)} * FROM {qi(schema)}.{qi(table)} WITH (NOLOCK) {w}',
                     f'Preview {schema}.{table}')

## Cell 6 — COLUMN PROFILER

*Teaching note.* One query per column rather than a single giant `UNION ALL`. It is more
round trips, but a `UNION ALL` needs every column cast to a common type, and one column
that will not cast kills the whole profile. Per-column with `try/except` degrades to
"this column failed" and keeps the other 19. On an exploration bench, robustness beats
speed.

Percentiles are reported for numeric columns because CLAUDE.md asks for distributions
(P25/median/P75), not just means — a mean hides the 2,600-hour aggregate rows that make
`PLC_EMPLOYEE_HOURS` misleading.

In [ ]:
_NUMERIC_TYPES = {'int','bigint','smallint','tinyint','decimal','numeric','float','real','money','smallmoney'}
_DATE_TYPES    = {'date','datetime','datetime2','smalldatetime','datetimeoffset'}

def profile_table(table=None, schema=None, cols=None, where=None):
    """Null rate, distinct count, min/max for every column; percentiles for numerics."""
    table  = table or TARGET_TABLE
    schema = schema or TARGET_SCHEMA
    meta = describe_table(table, schema)
    if cols:
        meta = meta[meta['column_name'].isin(cols)]
    if len(meta) > PROFILE_MAX_COLS:
        print(f'*** {len(meta)} columns > PROFILE_MAX_COLS={PROFILE_MAX_COLS}. '
              f'Pass cols=[...] to narrow, or raise the cap in Cell 3.')
        meta = meta.head(PROFILE_MAX_COLS)
    src = f'{qi(schema)}.{qi(table)} WITH (NOLOCK)'
    w = f'WHERE {where}' if where else ''
    total = int(run_query(f'SELECT COUNT(*) AS n FROM {src} {w}', '').iloc[0, 0])
    print(f'{schema}.{table}: {total:,} rows{" (filtered)" if where else ""}\n')

    out = []
    for r in meta.itertuples(index=False):
        c, dt = r.column_name, str(r.data_type).lower()
        row = {'column': c, 'type': dt, 'rows': total}
        try:
            if dt in _NUMERIC_TYPES:
                q = f"""
SELECT COUNT(*) - COUNT({qi(c)}) AS nulls, COUNT(DISTINCT {qi(c)}) AS distinct_vals,
       MIN(CAST({qi(c)} AS FLOAT)) AS min_v, MAX(CAST({qi(c)} AS FLOAT)) AS max_v,
       AVG(CAST({qi(c)} AS FLOAT)) AS mean_v,
       PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY CAST({qi(c)} AS FLOAT)) OVER () AS p25,
       PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY CAST({qi(c)} AS FLOAT)) OVER () AS p50,
       PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY CAST({qi(c)} AS FLOAT)) OVER () AS p75
FROM {src} {w}
"""
                d = run_query(q, '').iloc[0].to_dict()
            else:
                q = f"""
SELECT COUNT(*) - COUNT({qi(c)}) AS nulls, COUNT(DISTINCT {qi(c)}) AS distinct_vals,
       CAST(MIN({qi(c)}) AS NVARCHAR(60)) AS min_v, CAST(MAX({qi(c)}) AS NVARCHAR(60)) AS max_v
FROM {src} {w}
"""
                d = run_query(q, '').iloc[0].to_dict()
            row.update(d)
            row['null_pct'] = round(float(d['nulls']) / max(total, 1) * 100, 2)
        except Exception as e:
            row['error'] = str(e)[:90]
        out.append(row)
    cols_order = ['column','type','rows','nulls','null_pct','distinct_vals','min_v','max_v',
                  'mean_v','p25','p50','p75','error']
    res = pd.DataFrame(out)
    return res[[c for c in cols_order if c in res.columns]]

def top_values(column, table=None, schema=None, n=None, where=None):
    """Most common values in a column — the fastest way to see whether something is a
    category, an identifier, or free text."""
    table  = table or TARGET_TABLE
    schema = schema or TARGET_SCHEMA
    n = n or TOP_VALUES_N
    w = f'WHERE {where}' if where else ''
    return run_query(f"""
SELECT TOP {int(n)} {qi(column)} AS value, COUNT(*) AS rows_n
FROM {qi(schema)}.{qi(table)} WITH (NOLOCK) {w}
GROUP BY {qi(column)}
ORDER BY COUNT(*) DESC
""", f'Top values: {column}')

## Cell 7 — `dbo.PLC_EMPLOYEE_HOURS` — STRUCTURE

The table the ops dashboard already reads for labor hours (`PLC_TABLE` in its Cell 3.5).
This section documents what is actually in it. Known traps carried over from
ops_dashboard v1.3.0 — confirm each still holds when you run this:

1. **709 aggregate garbage rows** with NULL `Department_Name`/`Employee_Name` carrying
   2,600–3,100 hours each (~1.05M hours, more than every real row combined).
2. **The feed re-loads overlapping windows**, so an employee-day can repeat (worst
   observed: 30 copies of 2026-06-22). Always de-duplicate.
3. **`Reg_Hrs` / `OT1_Hrs` / `OT2_Hrs` / `Paid_Hrs` / `Est_*` are pay-period values
   repeated on every daily row** (~14× the daily `Hours` total). Only `[Hours]` is trusted.
4. **Future-dated rows** exist (through 2026-08-19 against an AS_OF of 2026-08-13).

In [ ]:
plc_cols = describe_table()
print(f'\n=== {TARGET_SCHEMA}.{TARGET_TABLE} — {len(plc_cols)} columns ===')
print(plc_cols.to_string(index=False))

# Which columns does the dashboard actually read? Anything outside this set is either
# unused or (see the pay-period columns) deliberately quarantined.
_DASH_USES = ['ID','Employee_Name','WorkDate','Hours','Pay_Type','Department_Name',
              'Location_Name','SystemUpdatedDate','Reg_Hrs','OT1_Hrs']
_present = set(plc_cols['column_name'])
print('\nColumns the dashboard reads but that are MISSING here:',
      sorted(set(_DASH_USES) - _present) or 'none — extract is consistent with the feed')
print('Columns present that the dashboard never reads:',
      sorted(_present - set(_DASH_USES)))

# A pay-period column would settle the payroll-week question outright (see §8). Look for
# one before inferring anything.
_period_like = [c for c in _present
                if re.search(r'period|week|begin|start|end|check|pay_?date', str(c), re.I)]
print('\nCandidate pay-period / week columns:', _period_like or 'NONE — §8 must infer the week')

In [ ]:
prof_plc = profile_table()
print(prof_plc.to_string(index=False))

In [ ]:
# Categorical shape of the columns that drive scope decisions.
for _c in ['Pay_Type', 'Department_Name', 'Location_Name']:
    if _c in set(plc_cols['column_name']):
        print(f'\n=== {_c} ===')
        print(top_values(_c).to_string(index=False))

# TRAP 1 made visible: the aggregate rows. If this returns rows, the department filter in
# the dashboard extract is load-bearing and must stay.
print('\n=== Aggregate garbage rows (NULL Department_Name) ===')
print(run_query(f"""
SELECT COUNT(*) AS rows_n, SUM(CAST([Hours] AS FLOAT)) AS total_hours,
       MIN(CAST([Hours] AS FLOAT)) AS min_hours, MAX(CAST([Hours] AS FLOAT)) AS max_hours
FROM {qi(TARGET_SCHEMA)}.{qi(TARGET_TABLE)} WITH (NOLOCK)
WHERE Department_Name IS NULL OR Employee_Name IS NULL
""", 'NULL-department rows').to_string(index=False))

## Cell 8 — PAYROLL WEEK BOUNDARY PROBE  *(which weekday does the Paylocity week start?)*

The ops dashboard infers overtime FLSA-style as hours over 40 in a workweek, so **the
workweek's start day changes who has overtime and how much.** v1.3.0 assumed Sun–Sat.
This section determines the real boundary from the data instead of assuming either one.

Three independent tests, strongest first:

- **Test A — pay-period run detection.** `Reg_Hrs` is a *pay-period* value repeated on
  every daily row (trap 3 — the thing that makes it useless as a metric makes it perfect
  here). For one employee, the set of consecutive dates sharing a single `Reg_Hrs` value
  *is* a pay period. The day-of-week its runs begin on is the period start.
- **Test B — recorded-OT fit.** Where `OT1_Hrs` is non-zero (before 2026-02, after which
  it collapses to 0), recompute inferred OT under all seven candidate week starts and see
  which reproduces the recorded figure most closely. Best fit wins.
- **Test C — worked-day density.** A payroll week that starts Thursday usually shows a
  distinctive Thu-anchored rhythm in hours-per-weekday. Weakest of the three; corroboration
  only.

*Teaching note.* Test A is the one to trust: it reads a boundary the payroll system itself
wrote, rather than inferring one from a metric we already know is broken. B and C exist so
that a single odd result in A does not get taken on faith.

In [ ]:
# ── Pull the raw material once: daily rows with a deterministic day-of-week ──────────
# (DATEDIFF(day,'1900-01-01',d) % 7) is Monday=0..Sunday=6 and is immune to @@DATEFIRST,
# which is session-scoped and has bitten this codebase before.
DOW_NAMES = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

df_probe = run_query(f"""
SELECT TRIM(ISNULL(P.ID,''))              AS plc_id,
       P.Employee_Name                    AS employee_name,
       CAST(P.WorkDate AS DATE)           AS workdate,
       CAST(P.[Hours] AS FLOAT)           AS hours,
       ISNULL(P.Pay_Type,'(blank)')       AS pay_type,
       CAST(ISNULL(P.Reg_Hrs, 0) AS FLOAT) AS reg_hrs,
       CAST(ISNULL(P.OT1_Hrs, 0) AS FLOAT) AS ot1_hrs,
       (DATEDIFF(day, '1900-01-01', CAST(P.WorkDate AS DATE)) % 7) AS dow_mon0
FROM {qi(TARGET_SCHEMA)}.{qi(TARGET_TABLE)} AS P WITH (NOLOCK)
WHERE P.Department_Name IS NOT NULL
  AND P.Employee_Name IS NOT NULL
  AND P.WorkDate >= '{FILTER_START}'
  AND P.WorkDate <  DATEADD(day, 1, '{FILTER_END}')
""", 'PLC week-boundary probe')

df_probe['workdate'] = pd.to_datetime(df_probe['workdate'], errors='coerce')
df_probe = df_probe[df_probe['workdate'].notna()].copy()
# Same de-duplication rule the dashboard uses — repeated loads would fake period runs.
_before = len(df_probe)
df_probe = df_probe.drop_duplicates(
    subset=['plc_id','employee_name','workdate','pay_type','hours']).copy()
print(f'  de-duplicated {_before - len(df_probe):,} repeated rows -> {len(df_probe):,} kept')

In [ ]:
# ── TEST A — pay-period run detection ────────────────────────────────────────────────
# For each employee, walk their dates in order and start a new "run" whenever the
# repeated Reg_Hrs value changes. The first date of each run is a pay-period start.
_a = (df_probe[df_probe['reg_hrs'] > 0]
      .drop_duplicates(subset=['employee_name','workdate'])
      .sort_values(['employee_name','workdate']))

if _a.empty:
    print('TEST A: no non-zero Reg_Hrs rows — cannot detect period runs. Rely on B and C.')
    test_a_start = None
else:
    _g = _a.groupby('employee_name', sort=False)
    _new_emp    = _a['employee_name'].ne(_a['employee_name'].shift())
    _new_val    = _a['reg_hrs'].ne(_a['reg_hrs'].shift())
    _a['_run_id'] = (_new_emp | _new_val).cumsum()
    runs = (_a.groupby('_run_id')
              .agg(employee=('employee_name','first'), start=('workdate','min'),
                   end=('workdate','max'), days=('workdate','nunique'),
                   reg_hrs=('reg_hrs','first')))
    # A genuine pay period is a contiguous block. Drop 1-day runs (noise) and any run
    # longer than 16 days (an employee whose Reg_Hrs happened not to change across two
    # periods — indistinguishable from one long run, so excluded rather than guessed).
    runs = runs[(runs['days'] >= 3) & ((runs['end'] - runs['start']).dt.days <= 16)]
    runs['start_dow'] = runs['start'].dt.dayofweek          # Monday=0
    runs['span_days'] = (runs['end'] - runs['start']).dt.days + 1
    _dist = (runs['start_dow'].value_counts(normalize=True).sort_index() * 100).round(1)
    _tbl = pd.DataFrame({'weekday': [DOW_NAMES[i] for i in _dist.index],
                         'pct_of_period_starts': _dist.values,
                         'n': runs['start_dow'].value_counts().sort_index().values})
    print(f'TEST A — {len(runs):,} pay-period runs detected across '
          f'{runs["employee"].nunique():,} employees')
    print(_tbl.to_string(index=False))
    print(f'\n  run length: median {runs["span_days"].median():.0f} days '
          f'(P25 {runs["span_days"].quantile(.25):.0f} / P75 {runs["span_days"].quantile(.75):.0f}) '
          f'-> {"weekly" if runs["span_days"].median() <= 8 else "biweekly"} periods')
    test_a_start = int(runs['start_dow'].mode().iloc[0])
    _share = float(_dist.max())
    print(f'  => period starts on {DOW_NAMES[test_a_start]} '
          f'({_share:.1f}% of runs){"" if _share >= 60 else "  *** WEAK: no dominant weekday"}')

In [ ]:
# ── TEST B — recorded-OT fit across all seven candidate week starts ──────────────────
# OT1_Hrs is a pay-period value repeated per row, so compare on the pay-period TOTAL per
# employee (max of the repeated value), not a sum of daily rows.
_b = df_probe[df_probe['ot1_hrs'] > 0].copy()
if _b.empty:
    print('TEST B: OT1_Hrs is zero everywhere in the window — no recorded OT to fit against.')
    print('        (Expected: the dashboard documents OT1_Hrs collapsing to 0 from 2026-02.)')
    test_b_start = None
else:
    _b_max = _b['workdate'].max()
    print(f'TEST B — fitting against recorded OT1_Hrs; usable rows end {_b_max.date()}')
    _work = df_probe[df_probe['pay_type'].isin(['Work','On Call Hours'])].copy()
    _work = _work[_work['workdate'] <= _b_max]
    _recorded = (_b.groupby('employee_name')['ot1_hrs'].max().rename('recorded_ot'))
    _fit = []
    for _start in range(7):                    # 0=Mon .. 6=Sun
        _w = _work.copy()
        _w['week_start'] = _w['workdate'] - pd.to_timedelta(
            (_w['dow_mon0'] - _start) % 7, unit='D')
        _wk = _w.groupby(['employee_name','week_start'], as_index=False)['hours'].sum()
        _wk['ot'] = (_wk['hours'] - 40.0).clip(lower=0)
        _inf = _wk.groupby('employee_name')['ot'].sum().rename('inferred_ot')
        _j = pd.concat([_recorded, _inf], axis=1).dropna()
        if _j.empty:
            continue
        _mae = float((_j['inferred_ot'] - _j['recorded_ot']).abs().mean())
        _fit.append({'week_start_dow': DOW_NAMES[_start], 'mean_abs_error_hrs': round(_mae, 2),
                     'total_inferred': round(float(_j['inferred_ot'].sum()), 0),
                     'total_recorded': round(float(_j['recorded_ot'].sum()), 0),
                     'employees': len(_j)})
    _fitdf = pd.DataFrame(_fit).sort_values('mean_abs_error_hrs').reset_index(drop=True)
    print(_fitdf.to_string(index=False))
    test_b_start = DOW_NAMES.index(_fitdf.iloc[0]['week_start_dow'])
    print(f'  => best fit: week starts {_fitdf.iloc[0]["week_start_dow"]} '
          f'(MAE {_fitdf.iloc[0]["mean_abs_error_hrs"]} h/employee)')
    if len(_fitdf) > 1 and (_fitdf.iloc[1]['mean_abs_error_hrs']
                            - _fitdf.iloc[0]['mean_abs_error_hrs']) < 0.5:
        print('  *** WEAK: the top two candidates are within 0.5 h — not a decisive fit.')

In [ ]:
# ── TEST C — worked-hours rhythm by weekday (corroboration only) ─────────────────────
_c = (df_probe[df_probe['pay_type'].isin(['Work','On Call Hours'])]
      .groupby('dow_mon0').agg(hours=('hours','sum'), rows=('hours','size')))
_c['pct_of_hours'] = (_c['hours'] / _c['hours'].sum() * 100).round(1)
_c.index = [DOW_NAMES[i] for i in _c.index]
print('TEST C — worked hours by weekday')
print(_c.to_string())

# ── VERDICT ──────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 72)
print('WEEK BOUNDARY VERDICT')
print('=' * 72)
for _nm, _v in [('Test A (pay-period runs)', test_a_start),
                ('Test B (recorded-OT fit)', test_b_start)]:
    print(f'  {_nm:28} {DOW_NAMES[_v] if _v is not None else "inconclusive"}')
_votes = [v for v in (test_a_start, test_b_start) if v is not None]
if _votes and len(set(_votes)) == 1:
    print(f'\n  AGREED: the payroll week starts on {DOW_NAMES[_votes[0]]}.')
    print(f'  Set OT_WEEK_START_DOW = {_votes[0]} in ops_dashboard Cell 3.1.')
elif _votes:
    print('\n  *** TESTS DISAGREE. Do not change OT_WEEK_START_DOW on this evidence —')
    print('      confirm the workweek definition with payroll before publishing OT.')
else:
    print('\n  *** INCONCLUSIVE. Confirm the workweek definition with payroll directly.')
print('\n  Reference: Thu=3 (Thu-Wed week), Sun=6 (Sun-Sat, the v1.3.0 assumption).')

## Cell 9 — SCRATCH

Throwaway queries go here, not in the sections above. Anything worth keeping should
graduate into a named section with a teaching note, or into the dashboard.

In [ ]:
# e.g.
# list_tables(like='PLC%')
# table_sizes(like='PLC%')
# describe_table('SERP_APC_DAILY')
# profile_table('SERP_APC_DAILY', cols=['warehourse','total','date'])
# preview('PLC_EMPLOYEE_HOURS', n=3)

## Cell 10 — CLOSE CONNECTION

In [ ]:
try:
    sql_conn.close()
    print('DB connection closed.')
except Exception as _e:
    print(f'(connection already closed: {_e})')